In [1]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_current_matchup
from fantasy.nba_stats import get_games_this_week, batch_get_player_stats
from fantasy.analysis import project_player
from fantasy.llm import build_start_sit_prompt, ask_gemini
import pandas as pd

In [2]:
matchup = get_current_matchup()
week_start, week_end = matchup["week_start"], matchup["week_end"]
my_roster = get_my_roster()

with_stats = batch_get_player_stats(my_roster)
players = [{**p, "games_remaining": get_games_this_week(p["team_abbr"], week_start, week_end)}
           for p in with_stats]

# Sort by projected PTS descending
def projected_pts(p):
    if p["stats"] is None:
        return 0
    proj = project_player(p["stats"], p["games_remaining"], p.get("status", ""))
    return proj.get("PTS", 0)

players.sort(key=projected_pts, reverse=True)

In [3]:
from fantasy.analysis import project_player

for p in players:
    if p['stats'] is not None:
        p['projected'] = project_player(p['stats'], p['games_remaining'], p.get('status', ''))
    else:
        p['projected'] = {col: 0.0 for col in ['PTS','REB','AST','STL','BLK','TOV','FGM','FGA','FG3M','FTM','FTA']}

players.sort(key=lambda p: p['projected'].get('PTS', 0), reverse=True)

In [4]:
rows = []
for p in players:
    rows.append({
        "Player": p["name"],
        "Pos": p["position"],
        "Status": p["status"] or "Active",
        "Games Left": p["games_remaining"],
        "Proj PTS": round(p["projected"].get("PTS", 0), 1),
        "Proj REB": round(p["projected"].get("REB", 0), 1),
        "Proj AST": round(p["projected"].get("AST", 0), 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

               Player      Pos Status  Games Left  Proj PTS  Proj REB  Proj AST
         Kevin Durant SG,SF,PF Active           4     103.6      22.6      16.8
           RJ Barrett SG,SF,PF Active           4      80.3      21.5      11.5
        Jalen Brunson       PG Active           3      71.6      10.6      23.4
         Jrue Holiday    PG,SG Active           4      65.2      17.4      24.2
        Austin Reaves PG,SG,SF Active           3      64.4      12.8      15.9
          Evan Mobley     PF,C Active           3      58.5      28.0       7.9
         Jayson Tatum    SF,PF Active           3      57.3      26.7       9.9
       Onyeka Okongwu     PF,C Active           4      56.1      30.8      13.1
            Tre Jones    PG,SG Active           4      56.1      11.9      18.5
          Jalen Suggs    PG,SG      O           4      54.6      13.8      19.8
        Dyson Daniels PG,SG,SF Active           4      48.0      28.1      22.2
        Naji Marshall SG,SF,PF Active   

In [6]:
import importlib, os
from dotenv import load_dotenv
load_dotenv("/mnt/c/Users/louis/Documents/dev/nba_fantasy/.env", override=True)

prompt = build_start_sit_prompt(players)
advice = ask_gemini(prompt)
print("\n=== TODAY'S LINEUP CARD ===\n")
print(advice)


=== TODAY'S LINEUP CARD ===

Okay, here are my start/sit recommendations for each player, based on projected contribution and injury status:

1.  **Kevin Durant: START.** Durant is your top player with the highest projected points, making him an obvious start.
2.  **RJ Barrett: START.** Barrett's high projected score and four remaining games make him a must-start.
3.  **Jalen Brunson: START.** Brunson is a strong starting option with a high points projection over his three remaining games.
4.  **Jrue Holiday: START.** Holiday should be started given his solid projected points and four games left.
5.  **Austin Reaves: START.** Reaves' projected points make him a viable starter despite only having three games remaining.
6.  **Evan Mobley: START.** Mobley's a solid start despite having only three games left given his solid projected points.
7.  **Jayson Tatum: START.** Tatum should be started because of his scoring potential despite only having three games left.
8.  **Onyeka Okongwu: STA